In [5]:
import base64
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

FICHIER = "ventes.csv" if Path("ventes.csv").exists() else "ventes_sales.csv"

df_brut = pd.read_csv(
    FICHIER, sep=";", dtype=str, keep_default_na=False, encoding="latin1"
)
df_brut.insert(0, "ligne_fichier", df_brut.index + 2)
df = df_brut.copy()

journal_nettoyage = []


def tracer(etape, nb):
  journal_nettoyage.append({"etape": etape, "nb": int(nb)})
  print(f"{etape:<60} {int(nb):>7}")


print("Boyut:", df.shape)


def verifier(condition, message):
  print(("OK        " if condition else "À REVOIR  ") + message)


Boyut: (60880, 57)


In [6]:
COLONNES_TEXTE = [c for c in df.columns if c != "ligne_fichier"]


# UTF-8 ve ISO-8859-9 onarım fonksiyonları
def reparer_utf8(s):
  if not isinstance(s, str) or not s:
    return s
  for _ in range(3):
    if not re.search(r"Ã|Â", s):
      break
    repaired = False
    for enc in ("cp1252", "latin1"):
      try:
        candidate = s.encode(enc).decode("utf-8")
        s = candidate
        repaired = True
        break
      except Exception:
        pass
    if not repaired:
      break
  return s


def reparer_iso9_latin1(s):
  if not isinstance(s, str) or not s:
    return s
  try:
    return s.encode("latin1").decode("iso-8859-9")
  except Exception:
    return s


for col in COLONNES_TEXTE:
  mask_utf8 = df[col].str.contains(r"Ã|Â", regex=True, na=False)
  if mask_utf8.any():
    nb = mask_utf8.sum()
    df.loc[mask_utf8, col] = df.loc[mask_utf8, col].apply(reparer_utf8)
    tracer(f"Encodage UTF-8 : {col}", nb)

  mask_iso9 = df[col].str.contains(r"[þýÝðÞÐ]", regex=True, na=False)
  if mask_iso9.any():
    nb = mask_iso9.sum()
    df.loc[mask_iso9, col] = df.loc[mask_iso9, col].apply(reparer_iso9_latin1)
    tracer(f"Encodage ISO-8859-9 : {col}", nb)

# SousCategorie içindeki kayıp (?) karakterleri geçerli kümeden regex ile tamamlama
valides_sc = set(df["SousCategorie"].dropna())
valides_sc = {
    v
    for v in valides_sc
    if not re.search(r"\w\?|\?\w", v) and not re.search(r"Ã|Â", v)
}

mask_perdus = df["SousCategorie"].str.contains(r"\w\?|\?\w", regex=True, na=False)
if mask_perdus.any():
  nb = mask_perdus.sum()

  def reparer_perdu(val):
    pattern = "^" + "".join("." if ch == "?" else re.escape(ch) for ch in val) + "$"
    matches = [v for v in valides_sc if re.match(pattern, v)]
    return matches[0] if len(matches) == 1 else val

  df.loc[mask_perdus, "SousCategorie"] = df.loc[
      mask_perdus, "SousCategorie"
  ].apply(reparer_perdu)
  tracer("Caractères perdus (?) : SousCategorie", nb)

# Kontrol
suspects = (
    df[COLONNES_TEXTE]
    .apply(lambda s: s.str.contains(r"Ã|Â|[þýÝðÞÐ]", regex=True, na=False))
    .sum()
    .sum()
)
verifier(suspects == 0, f"plus de séquence mal encodée ({suspects} restantes)")
perdus = (
    df["SousCategorie"].str.contains(r"\w\?|\?\w", regex=True, na=False).sum()
)
verifier(
    perdus == 0,
    f"plus de caractère perdu dans SousCategorie ({perdus} restants)",
)


Encodage UTF-8 : Prenom                                          151
Encodage UTF-8 : Nom                                             195
Encodage UTF-8 : Email                                           149
Encodage UTF-8 : AdresseLigne1                                  2455
Encodage UTF-8 : AdresseLigne2                                   618
Encodage UTF-8 : Education                                     16655
Encodage UTF-8 : Profession                                     9528
Encodage UTF-8 : Ville                                           658
Encodage UTF-8 : Pays                                          20889
Encodage ISO-8859-9 : Pays                                       123
Encodage UTF-8 : SousCategorie                                 27615
Encodage UTF-8 : Categorie                                     24367
Encodage ISO-8859-9 : PrixCatalogue                             3202
Encodage UTF-8 : Promotion                                      2121
Encodage UTF-8 : Devise           

In [7]:
MARQUEURS_MANQUANTS = ["", "NULL", "N/A", "?", "inconnu", "-"]

# Couleur sütunundaki 'NA' (renksiz) özel anlam taşıdığı için korunur
for col in COLONNES_TEXTE:
  if col == "Couleur":
    mask = df[col].isin([m for m in MARQUEURS_MANQUANTS if m != "NA"])
  else:
    mask = df[col].isin(MARQUEURS_MANQUANTS)
  nb = mask.sum()
  if nb > 0:
    df.loc[mask, col] = np.nan
    tracer(f"Valeurs manquantes : {col}", nb)

# Kontrol
restants = (
    df[COLONNES_TEXTE]
    .isin(["", "NULL", "N/A", "?", "inconnu", "-"])
    .sum()
    .sum()
)
verifier(
    restants == 0,
    f"plus aucun marqueur de valeur manquante ({restants} restants)",
)
verifier(
    (df["Couleur"] == "NA").sum() > 0,
    "la couleur « NA » (sans couleur) a été conservée",
)


Valeurs manquantes : IdClient                                    603
Valeurs manquantes : Civilite                                  59966
Valeurs manquantes : DeuxiemePrenom                            25705
Valeurs manquantes : Suffixe                                   60874
Valeurs manquantes : DateNaissance                               302
Valeurs manquantes : Email                                      1213
Valeurs manquantes : Telephone                                   304
Valeurs manquantes : AdresseLigne1                               304
Valeurs manquantes : AdresseLigne2                             58865
Valeurs manquantes : RevenuAnnuel                                304
Valeurs manquantes : NbEnfants                                   304
Valeurs manquantes : NbEnfantsDomicile                           304
Valeurs manquantes : Education                                   304
Valeurs manquantes : Profession                                  912
Valeurs manquantes : Proprietaire 

In [8]:
# 3. Birebir kopyaları silme
contenu = [c for c in df.columns if c != "ligne_fichier"]
nb_doublons = df.duplicated(subset=contenu).sum()
df = df.drop_duplicates(subset=contenu, keep="first").copy()
tracer("Doublons exacts supprimés", nb_doublons)
verifier(not df.duplicated(subset=contenu).any(), "plus aucun doublon exact")

# 4. Anahtar çakışmaları ve Rejets
CLE = ["NumCommande", "NumLigne"]
cols_hors_cle = [c for c in df.columns if c not in CLE and c != "ligne_fichier"]
df["_empreinte"] = pd.util.hash_pandas_object(df[cols_hors_cle], index=False)

while True:
  conflits = df.duplicated(subset=CLE, keep=False)
  if not conflits.any():
    break
  empreintes_saines = set(df.loc[~conflits, "_empreinte"])
  parasites = conflits & df["_empreinte"].isin(empreintes_saines)
  if not parasites.any():
    break
  nb_parasites = parasites.sum()
  df = df.loc[~parasites].copy()
  tracer("Lignes parasites supprimées", nb_parasites)

conflits_finaux = df.duplicated(subset=CLE, keep=False)
rejets = df.loc[conflits_finaux].copy()
df = df.loc[~conflits_finaux].copy()
tracer("Lignes en conflit vers rejets", len(rejets))

df = df.drop(columns=["_empreinte"])
rejets = rejets.drop(columns=["_empreinte"], errors="ignore")

# Kontrol
verifier(
    not df.duplicated(subset=CLE).any(),
    "chaque clé (NumCommande, NumLigne) est unique dans df",
)
verifier(
    len(rejets) <= 0.01 * len(df),
    f"au plus 1 % des lignes en rejet ({len(rejets)})",
)
verifier(
    "_empreinte" not in df.columns,
    "la colonne technique _empreinte a été retirée de df",
)


Doublons exacts supprimés                                        181
OK        plus aucun doublon exact
Lignes parasites supprimées                                      300
Lignes parasites supprimées                                        1
Lignes en conflit vers rejets                                      0
OK        chaque clé (NumCommande, NumLigne) est unique dans df
OK        au plus 1 % des lignes en rejet (0)
OK        la colonne technique _empreinte a été retirée de df


In [9]:
# 1. Telephone -> Telephone2
df["Telephone2"] = np.nan
mask_tel = df["Telephone"].str.contains(" / ", na=False)
if mask_tel.any():
  split_tel = df.loc[mask_tel, "Telephone"].str.split(" / ", n=1, expand=True)
  df.loc[mask_tel, "Telephone"] = split_tel[0]
  df.loc[mask_tel, "Telephone2"] = split_tel[1]
  tracer("Multivalué Telephone", mask_tel.sum())

# 2. Email -> EmailSecondaire
df["EmailSecondaire"] = np.nan
mask_mail = df["Email"].str.contains(";", na=False)
if mask_mail.any():
  split_mail = df.loc[mask_mail, "Email"].str.split(r";\s*", n=1, expand=True)
  df.loc[mask_mail, "Email"] = split_mail[0]
  df.loc[mask_mail, "EmailSecondaire"] = split_mail[1]
  tracer("Multivalué Email", mask_mail.sum())

# 3. Couleur birden fazla olanları boşalt (daha sonra ürün kodundan tamamlanacak)
mask_coul = df["Couleur"].str.contains(r"/|,|&", regex=True, na=False)
if mask_coul.any():
  df.loc[mask_coul, "Couleur"] = np.nan
  tracer("Multivalué Couleur vidée", mask_coul.sum())

# Kontrol
verifier(
    {"Telephone2", "EmailSecondaire"} <= set(df.columns),
    "colonnes Telephone2 et EmailSecondaire créées",
)
verifier(
    not df["Telephone"].str.contains(" / ", na=False).any(),
    "un seul numéro par cellule Telephone",
)
verifier(
    not df["Email"].str.contains(";", na=False).any(),
    "une seule adresse par cellule Email",
)
verifier(
    not df["Couleur"].str.contains(r"/|,|&", regex=True, na=False).any(),
    "une seule couleur par cellule",
)


/tmp/ipykernel_540/1033490025.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['621-555-3860' '998-555-3132' '347-555-8113' '569-555-9158'
 '334-555-3828' '879-555-6462' '627-555-9252' '721-555-0435'
 '198-555-6715' '474-555-4490' '513-555-4778' '316-555-7604'
 '450-555-1185' '334-555-0335' '941-555-1809' '162-555-7400'
 '234-555-4509' '271-555-7657' '684-555-1735' '208-555-7139'
 '155-555-9936' '760-555-2678' '297-555-0576' '483-555-3897'
 '895-555-0415' '162-555-8435' '304-555-9284' '799-555-6402'
 '149-555-8712' '207-555-4597' '493-555-0252' '189-555-0992'
 '631-555-2593' '167-555-0445' '540-555-5794' '778-555-9906'
 '537-555-2369' '345-555-2295' '361-555-2977' '617-555-2107'
 '170-555-9790' '524-555-0319' '865-555-7020' '167-555-5331'
 '982-555-9286' '792-555-9338' '369-555-5476' '539-555-6540'
 '810-555-3301' '389-555-4688' '284-555-2982' '708-555-0806'
 '964-555-1342' '567-555-1586' '873-555-92

Multivalué Telephone                                             603
Multivalué Couleur vidée                                         600
OK        colonnes Telephone2 et EmailSecondaire créées
OK        un seul numéro par cellule Telephone
OK        une seule adresse par cellule Email
OK        une seule couleur par cellule


In [10]:
def decoder_base64(s):
  if not isinstance(s, str) or "@" in s:
    return s
  try:
    dec = base64.b64decode(s, validate=True).decode("utf-8")
    if "@" in dec:
      return dec
  except Exception:
    pass
  return s


mask_b64 = (
    df["Email"].notna()
    & ~df["Email"].str.contains("@", na=False)
    & df["Email"].str.fullmatch(r"[A-Za-z0-9+/=]+", na=False)
)
if mask_b64.any():
  df.loc[mask_b64, "Email"] = df.loc[mask_b64, "Email"].apply(decoder_base64)
  tracer("Email décode Base64", mask_b64.sum())

# Hex hash içeren telefonları boşaltıyoruz
mask_hache = df["Telephone"].str.fullmatch(r"[0-9a-f]{16}", na=False)
if mask_hache.any():
  df.loc[mask_hache, "Telephone"] = np.nan
  tracer("Telephone empreinte vidée", mask_hache.sum())

# Kontrol
sans_arobase = (
    df["Email"].notna() & ~df["Email"].str.contains("@", na=False)
).sum()
verifier(
    sans_arobase == 0,
    f"tous les e-mails renseignés contiennent un @ ({sans_arobase} restants)",
)
haches = df["Telephone"].str.fullmatch(r"[0-9a-f]{16}", na=False).sum()
verifier(haches == 0, f"plus aucun téléphone haché ({haches} restants)")


Email décode Base64                                              905
Telephone empreinte vidée                                        601
OK        tous les e-mails renseignés contiennent un @ (0 restants)
OK        plus aucun téléphone haché (0 restants)


In [11]:
# 1. AdresseLigne1 sonundaki ", <CodePostal> <ville>" bilgisini temizleme
def nettoyer_adresse(row):
  a = row["AdresseLigne1"]
  cp = row["CodePostal"]
  if isinstance(a, str) and isinstance(cp, str) and cp:
    pat = r",\s*" + re.escape(cp) + r"(\s+.*|$)"
    if re.search(pat, a):
      return re.sub(pat, "", a).strip()
  return a


df["AdresseLigne1"] = df.apply(nettoyer_adresse, axis=1)

# 2. AdresseLigne2 içindeki telefonları taşıma
mask_addr_tel = df["AdresseLigne2"].str.startswith("Tél.", na=False)
if mask_addr_tel.any():
  tel_ext = df.loc[mask_addr_tel, "AdresseLigne2"].str.replace(
      r"^Tél\.\s*", "", regex=True
  )
  df.loc[mask_addr_tel & df["Telephone"].isna(), "Telephone"] = tel_ext
  df.loc[mask_addr_tel, "AdresseLigne2"] = np.nan
  tracer("AdresseLigne2 téléphone déplacé", mask_addr_tel.sum())

# 3. AdresseLigne2 içindeki şehirleri Ville sütununa taşıma
villes_connues = set(df["Ville"].dropna())
mask_addr_ville = df["Ville"].isna() & df["AdresseLigne2"].isin(villes_connues)
if mask_addr_ville.any():
  df.loc[mask_addr_ville, "Ville"] = df.loc[mask_addr_ville, "AdresseLigne2"]
  df.loc[mask_addr_ville, "AdresseLigne2"] = np.nan
  tracer("AdresseLigne2 ville déplacée", mask_addr_ville.sum())

# Kontrol
suffixe = sum(
    isinstance(a, str)
    and isinstance(cp, str)
    and re.search(r",\s*" + re.escape(cp) + r"(\s|$)", a) is not None
    for a, cp in zip(df["AdresseLigne1"], df["CodePostal"])
)
verifier(
    suffixe == 0,
    f"plus de ville ni de code postal dans AdresseLigne1 ({suffixe} restants)",
)
verifier(
    not df["AdresseLigne2"].str.startswith("Tél.", na=False).any(),
    "plus de téléphone dans AdresseLigne2",
)
villes = set(df["Ville"].dropna())
deplacer = (df["Ville"].isna() & df["AdresseLigne2"].isin(villes)).sum()
verifier(
    deplacer == 0, f"plus de ville saisie dans AdresseLigne2 ({deplacer} restantes)"
)


AdresseLigne2 téléphone déplacé                                  603
AdresseLigne2 ville déplacée                                     300
OK        plus de ville ni de code postal dans AdresseLigne1 (0 restants)
OK        plus de téléphone dans AdresseLigne2
OK        plus de ville saisie dans AdresseLigne2 (0 restantes)


In [12]:
# 8.1 Tarihler
COLONNES_DATE = [
    "DateCommande",
    "DateEcheance",
    "DateExpedition",
    "DateNaissance",
    "DatePremierAchat",
]
FORMATS_DATE = [
    "%Y-%m-%d",
    "%Y%m%d",
    "%d.%m.%Y",
    "%d/%m/%Y",
    "%b %d, %Y",
    "%m/%d/%Y",
]


def convertir_dates(serie):
  res = pd.Series(pd.NaT, index=serie.index)
  for fmt in FORMATS_DATE:
    manquants = res.isna() & serie.notna()
    if not manquants.any():
      break
    res.loc[manquants] = pd.to_datetime(
        serie.loc[manquants], format=fmt, errors="coerce"
    )
  return res


for col in COLONNES_DATE:
  df[col] = convertir_dates(df[col])

# 8.2 Sayılar
COLONNES_NOMBRE = [
    "RevenuAnnuel",
    "NbEnfants",
    "NbEnfantsDomicile",
    "NbVoitures",
    "Poids",
    "PrixCatalogue",
    "TauxRemisePromo",
    "Quantite",
    "PrixUnitaire",
    "TauxRemise",
    "MontantRemise",
    "CoutStandard",
    "CoutTotal",
    "MontantVente",
    "Taxe",
    "FraisPort",
]


def convertir_nombre(s):
  if not isinstance(s, str) or not s.strip():
    return np.nan
  s = s.replace(" ", "").replace("\xa0", "")
  if "," in s and "." in s:
    s = s.replace(",", "")
  elif "," in s:
    s = s.replace(",", ".")
  try:
    return float(s)
  except Exception:
    return np.nan


for col in COLONNES_NOMBRE:
  df[col] = df[col].apply(convertir_nombre)

# 8.3 Telefonlar
FORMAT_TEL = r"\d{3}-\d{3}-\d{4}|\d \(\d\d\) \d{3} \d{3}-\d{4}"


def normaliser_telephone(s):
  if not isinstance(s, str) or not s.strip():
    return np.nan
  if re.fullmatch(r"\d \(\d\d\) \d{3} \d{3}-\d{4}", s):
    return s
  chiffres = re.sub(r"\D", "", s)
  if len(chiffres) == 11 and chiffres.startswith("1"):
    chiffres = chiffres[1:]
  if len(chiffres) == 10:
    return f"{chiffres[:3]}-{chiffres[3:6]}-{chiffres[6:]}"
  return s


for col in ["Telephone", "Telephone2"]:
  df[col] = df[col].apply(normaliser_telephone)

# Kontroller
for col in COLONNES_DATE:
  est_date = pd.api.types.is_datetime64_any_dtype(df[col])
  attendues = (
      df_brut.loc[df.index, col]
      .replace(MARQUEURS_MANQUANTS, np.nan)
      .notna()
      .sum()
  )
  lues = df[col].notna().sum() if est_date else 0
  verifier(
      est_date and lues >= attendues,
      f"{col} : type date, aucune date illisible ({lues}/{attendues})",
  )

non_num = [c for c in COLONNES_NOMBRE if not pd.api.types.is_float_dtype(df[c])]
verifier(
    not non_num,
    f"toutes les colonnes numériques sont des float {non_num if non_num else ''}",
)

hors = (~df["Telephone"].str.fullmatch(FORMAT_TEL, na=True)).sum()
verifier(
    hors <= 0.001 * len(df),
    f"au moins 99,9 % des téléphones au format standard ({hors} hors format)",
)


OK        DateCommande : type date, aucune date illisible (60398/60398)
OK        DateEcheance : type date, aucune date illisible (60398/60398)
OK        DateExpedition : type date, aucune date illisible (60398/60398)
OK        DateNaissance : type date, aucune date illisible (60097/60097)
OK        DatePremierAchat : type date, aucune date illisible (60097/60097)
OK        toutes les colonnes numériques sont des float 
OK        au moins 99,9 % des téléphones au format standard (5 hors format)


In [13]:
CORRESPONDANCES = {
    "Pays": {
        "United States": "États-Unis",
        "US": "États-Unis",
        "USA": "États-Unis",
        "Amerika Birleşik Devletleri": "États-Unis",
        "United Kingdom": "Royaume-Uni",
        "UK": "Royaume-Uni",
        "Birleşik Krallık": "Royaume-Uni",
        "Germany": "Allemagne",
        "Deutschland": "Allemagne",
        "Almanya": "Allemagne",
        "Australia": "Australie",
        "Avustralya": "Australie",
        "Canada": "Canada",
        "Kanada": "Canada",
        "France": "France",
        "Fransa": "France",
    },
    "Devise": {
        "US Dollar": "USD",
        "Dollar US": "USD",
        "$": "USD",
        "US$": "USD",
        "Euro": "EUR",
        "€": "EUR",
        "British Pound": "GBP",
        "Livre Sterling": "GBP",
        "£": "GBP",
        "Australian Dollar": "AUD",
        "Dollar australien": "AUD",
        "AU$": "AUD",
        "A$": "AUD",
        "Canadian Dollar": "CAD",
        "Dollar canadien": "CAD",
        "C$": "CAD",
        "CAN$": "CAD",
        "FF": "FRF",
        "Franc": "FRF",
        "DM": "DEM",
    },
    "Genre": {
        "Homme": "M",
        "Male": "M",
        "Man": "M",
        "H": "M",
        "1": "M",
        "Femme": "F",
        "Female": "F",
        "Woman": "F",
        "W": "F",
        "2": "F",
    },
    "Proprietaire": {
        "True": "1",
        "TRUE": "1",
        "true": "1",
        "Oui": "1",
        "Yes": "1",
        "Y": "1",
        "Vrai": "1",
        "1.0": "1",
        1: "1",
        1.0: "1",
        "False": "0",
        "FALSE": "0",
        "false": "0",
        "Non": "0",
        "No": "0",
        "N": "0",
        "Faux": "0",
        "0.0": "0",
        0: "0",
        0.0: "0",
    },
}

for col, table in CORRESPONDANCES.items():
  avant = df[col].copy()
  df[col] = df[col].astype(str).replace("nan", np.nan).replace(table)
  # Geçersiz tek tük semboller varsa temizle
  if col == "Genre":
    df.loc[~df["Genre"].isin(["M", "F"]), "Genre"] = np.nan
  tracer(
      f"Harmonisation codes : {col}", (avant.notna() & (avant != df[col])).sum()
  )

# Ülkeyi CodePays ile çapraz eşitleme
pays_maj = df.groupby("CodePays")["Pays"].agg(
    lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan
)
df["Pays"] = df["CodePays"].map(pays_maj)

# Kontrol
PAYS_REF = {
    "Allemagne",
    "Australie",
    "Canada",
    "France",
    "Royaume-Uni",
    "États-Unis",
}
verifier(
    set(df["Pays"].dropna()) <= PAYS_REF,
    f"Pays : libellés de référence uniquement"
    f" {sorted(set(df['Pays'].dropna()) - PAYS_REF)}",
)
verifier(
    df["Devise"].dropna().str.fullmatch(r"[A-Z]{3}").all(),
    "Devise : codes ISO à trois lettres uniquement",
)
verifier(set(df["Genre"].dropna()) <= {"M", "F"}, "Genre : M ou F uniquement")
verifier(
    set(df["Proprietaire"].dropna()) <= {"0", "1"},
    "Proprietaire : 0 ou 1 uniquement",
)
verifier(
    (df.groupby("CodePays")["Pays"].nunique() <= 1).all(),
    "un seul libellé de pays par CodePays",
)


Harmonisation codes : Pays                                      1096
Harmonisation codes : Devise                                    1133
Harmonisation codes : Genre                                     1811
Harmonisation codes : Proprietaire                              1198
OK        Pays : libellés de référence uniquement []
OK        Devise : codes ISO à trois lettres uniquement
OK        Genre : M ou F uniquement
OK        Proprietaire : 0 ou 1 uniquement
OK        un seul libellé de pays par CodePays


In [14]:
# 10.1 Eksik IdClient'ları tekil eşleşmelerle kurtarma
combinaisons = [
    ["Email"],
    ["Prenom", "DateNaissance", "Telephone"],
    ["Prenom", "DateNaissance", "CodePostal"],
]

for comb in combinaisons:
  connus = df[df["IdClient"].notna()].dropna(subset=comb)
  uniques = connus.groupby(comb)["IdClient"].nunique()
  valides_comb = uniques[uniques == 1].index
  ref = connus.set_index(comb).loc[valides_comb, "IdClient"].drop_duplicates()

  manquants = df["IdClient"].isna()
  if isinstance(ref.index, pd.MultiIndex):
    tuples = list(zip(*[df.loc[manquants, c] for c in comb]))
    retrouves = pd.Series(tuples, index=df[manquants].index).map(ref)
  else:
    retrouves = df.loc[manquants, comb[0]].map(ref)

  remplaces = retrouves.notna()
  df.loc[manquants & remplaces, "IdClient"] = retrouves[remplaces]
  tracer(f"IdClient retrouvés via {comb}", remplaces.sum())

initial = (
    df_brut.loc[df.index, "IdClient"]
    .isin(["", "NULL", "N/A", "?", "inconnu", "-"])
    .sum()
)
restants = df["IdClient"].isna().sum()
verifier(
    restants <= 0.1 * initial,
    f"au moins 90 % des IdClient manquants retrouvés ({restants} encore"
    " manquants)",
)

# 10.2 Çoğunluk Oyu (Voting) ve Doğrulama
VALIDE = {
    "Email": lambda r: bool(
        isinstance(r["Email"], str)
        and re.fullmatch(r"[^@\s]+@[^@\s]+\.\w+", r["Email"])
    ),
    "Telephone": lambda r: bool(
        isinstance(r["Telephone"], str) and re.fullmatch(FORMAT_TEL, r["Telephone"])
    ),
    "DateNaissance": lambda r: bool(
        pd.notna(r["DateNaissance"])
        and (
            pd.isna(r["DatePremierAchat"])
            or r["DateNaissance"] < r["DatePremierAchat"]
        )
    ),
    "Nom": lambda r: bool(
        isinstance(r["Nom"], str)
        and not r["Nom"].isupper()
        and not r["Nom"].islower()
        and not re.fullmatch(r"[A-Z]\.\s*\w+", r["Nom"])
        and not " " in r["Nom"]
    ),
    "Civilite": lambda r: bool(
        (r["Genre"] == "M" and r["Civilite"] in ["Mr.", "Sr."])
        or (r["Genre"] == "F" and r["Civilite"] in ["Ms.", "Ms", "Mrs.", "Sra."])
    ),
    "DateExpedition": lambda r: bool(
        pd.notna(r["DateExpedition"])
        and (
            pd.isna(r["DateCommande"])
            or r["DateExpedition"] >= r["DateCommande"]
        )
    ),
}


def consolider(df, cle, colonnes, etiquette):
  for col in colonnes:
    valide = df[col].notna() & df[cle].notna()
    if col in VALIDE:
      valide &= df.apply(VALIDE[col], axis=1)
    votes = (
        df[valide]
        .groupby([cle, col], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values([cle, "n"], ascending=[True, False])
        .drop_duplicates(cle)
        .set_index(cle)[col]
    )
    avant = df[col].copy()
    nouveau = df[cle].map(votes)
    a_remplacer = df[cle].isin(votes.index)
    df.loc[a_remplacer, col] = nouveau[a_remplacer]
    if col in VALIDE:
      invalide = df[col].notna() & ~a_remplacer & ~df.apply(VALIDE[col], axis=1)
      df.loc[invalide, col] = np.nan
    change = ~((avant == df[col]) | (avant.isna() & df[col].isna()))
    tracer(f"Consolidation {etiquette} : {col}", change.sum())
  return df


# 1. Commande
df = consolider(
    df,
    "NumCommande",
    ["DateCommande", "DateEcheance", "IdClient", "Devise"],
    "Commande",
)
df = consolider(df, "NumCommande", ["DateExpedition"], "Commande")

# 2. Client
df = consolider(df, "IdClient", ["Genre"], "Client")
df = consolider(
    df,
    "IdClient",
    [
        "Civilite",
        "Prenom",
        "DeuxiemePrenom",
        "Nom",
        "Suffixe",
        "DateNaissance",
        "EtatCivil",
        "Email",
        "Telephone",
        "AdresseLigne1",
        "AdresseLigne2",
        "RevenuAnnuel",
        "NbEnfants",
        "NbEnfantsDomicile",
        "Education",
        "Profession",
        "Proprietaire",
        "NbVoitures",
        "DistanceTrajet",
        "DatePremierAchat",
        "Ville",
        "CodeProvince",
        "Province",
        "CodePays",
        "Pays",
        "CodePostal",
    ],
    "Client",
)

# 3. Produit
# CodeProduit boş olan hayalet sipariş satırlarını eliyoruz
df = df.dropna(subset=["CodeProduit"]).copy()
df = consolider(
    df,
    "CodeProduit",
    ["Produit", "Couleur", "SousCategorie", "Categorie"],
    "Produit",
)

# Kontrol
verifier(
    (df.groupby("NumCommande")["DateCommande"].nunique() <= 1).all(),
    "une seule date par commande",
)
for col in ["Nom", "Email", "Telephone", "DateNaissance", "Ville"]:
  verifier(
      (df.groupby("IdClient")[col].nunique() <= 1).all(),
      f"une seule valeur de {col} par client",
  )
verifier(
    (df.groupby("CodeProduit")["Couleur"].nunique() <= 1).all(),
    "une seule couleur par produit",
)
verifier(df["Couleur"].notna().all(), "toutes les couleurs reconstituées")


IdClient retrouvés via ['Email']                                 565
IdClient retrouvés via ['Prenom', 'DateNaissance', 'Telephone']      10
IdClient retrouvés via ['Prenom', 'DateNaissance', 'CodePostal']       0
OK        au moins 90 % des IdClient manquants retrouvés (28 encore manquants)
Consolidation Commande : DateCommande                              0
Consolidation Commande : DateEcheance                              0
Consolidation Commande : IdClient                                  0
Consolidation Commande : Devise                                  318
Consolidation Commande : DateExpedition                          301
Consolidation Client : Genre                                     239
Consolidation Client : Civilite                                  744
Consolidation Client : Prenom                                      0
Consolidation Client : DeuxiemePrenom                              0
Consolidation Client : Nom                                       609
Consolidation Cli

In [15]:
montant_attendu = (df["Taxe"] / 0.08).round(2)

# Kural 1: Quantite <= 0 veya boş olanları hesaplama
mask_qte = df["Quantite"].isna() | (df["Quantite"] <= 0)
if mask_qte.any():
  qte_calculee = (
      (montant_attendu + df["MontantRemise"].fillna(0)) / df["PrixUnitaire"]
  ).round()
  df.loc[mask_qte, "Quantite"] = qte_calculee[mask_qte]
  tracer("Règle gestion : Quantite recalculée", mask_qte.sum())

# Kural 2: MontantVente tutarsızlıklarını formülle tamamlama
montant_theorique = (
    df["Quantite"] * df["PrixUnitaire"] - df["MontantRemise"].fillna(0)
).round(2)
mask_montant = df["MontantVente"].isna() | (
    (df["MontantVente"] - montant_theorique).abs() > 0.01
)
if mask_montant.any():
  df.loc[mask_montant, "MontantVente"] = montant_theorique[mask_montant]
  tracer("Règle gestion : MontantVente recalculé", mask_montant.sum())

# Kural 3: Sevkiyat tarihi (DateExpedition) kuralı ve medyan ataması
delai_valide = (df["DateExpedition"] - df["DateCommande"]).dt.days
delai_median = pd.Timedelta(days=int(delai_valide[delai_valide >= 0].median()))

mask_exped = df["DateExpedition"].isna() | (
    df["DateExpedition"] < df["DateCommande"]
)
df["DateExpedition_imputee"] = mask_exped
if mask_exped.any():
  df.loc[mask_exped, "DateExpedition"] = (
      df.loc[mask_exped, "DateCommande"] + delai_median
  )
  tracer("Règle gestion : DateExpedition imputée", mask_exped.sum())

verifier(
    "DateExpedition_imputee" in df.columns, "colonne DateExpedition_imputee créée"
)


Règle gestion : Quantite recalculée                              300
Règle gestion : MontantVente recalculé                         23802
Règle gestion : DateExpedition imputée                            46
OK        colonne DateExpedition_imputee créée


In [16]:
controles = {
    "Lignes": len(df),
    "Clés (NumCommande, NumLigne) dupliquées": df.duplicated(CLE).sum(),
    "IdClient manquants": df["IdClient"].isna().sum(),
    "Emails invalides": (
        ~df["Email"].str.fullmatch(r"[^@\s]+@[^@\s]+\.\w+", na=True)
    ).sum(),
    "Quantite <= 0": (df["Quantite"] <= 0).sum(),
    "Montant incohérent": (
        df["MontantVente"]
        - (df["Quantite"] * df["PrixUnitaire"] - df["MontantRemise"])
    )
    .abs()
    .gt(0.01)
    .sum(),
    "Expédition avant commande": (
        df["DateExpedition"] < df["DateCommande"]
    ).sum(),
    "Naissance après premier achat": (
        df["DateNaissance"] >= df["DatePremierAchat"]
    ).sum(),
    "Lignes en rejet": len(rejets),
}
print(pd.Series(controles, name="valeur").to_frame())

# Temizlenmiş CSV çıktısını alma
df_propre = df.sort_values(
    CLE, key=lambda s: s if s.name != "NumLigne" else s.astype(int)
)
df_propre.to_csv(
    "ventes_propres.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d"
)
rejets.to_csv("ventes_rejets.csv", index=False, encoding="utf-8")
print(
    f"Başarıyla dışa aktarıldı: {len(df_propre)} temiz satır, {len(rejets)} adet reddedilen satır."
)


                                         valeur
Lignes                                    60097
Clés (NumCommande, NumLigne) dupliquées       0
IdClient manquants                           27
Emails invalides                              0
Quantite <= 0                                 0
Montant incohérent                            0
Expédition avant commande                     0
Naissance après premier achat                 0
Lignes en rejet                               0
Başarıyla dışa aktarıldı: 60097 temiz satır, 0 adet reddedilen satır.
